In [ ]:
import ee
ee.Authenticate()
ee.Initialize()

In [ ]:
# Load AOI (Area of Interest)
sites = ee.FeatureCollection("users/hadicu06/Postdoc_Bonn/restor/UniversityOfBonn_onlySiteId_confidential")

# Load GAUL administrative areas
GAUL0 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")
GAUL1 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1")
GAUL2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")


In [ ]:
# Define parameters
params = {
    'numberOfControlUnits': 1000,
    'startDate': '2000-01-01',
    'endDate': '2023-12-31',
    'dataExtractionScale': 30,       ## this can be made different for different datasets e.g. for Landsat 30m, for other covariates that are coarser resolution adjust to the respective resolution; e.g. add argument to function extract_data_func_v3()
    'usePersistentLC': True,
    'projectAreaBuffer': 10000,
    'sceneOverallCloudThres': 30,
    'filterPerPixelCloud': True,
    'mergeSensors': True,
    'exportPerDate': False,                  ## not yet tested for True
    'annualReducer': ee.Reducer.mean(),
    'version': '',
    'scriptLink': '',
    'gdriveFolder': 'POSTDOC_BONN_GEE_v2',
    'datetimeFormat': 'YYYY-MM-dd'
}

In [ ]:
  ##### General helper functions ##################################

# Function to extract image properties and set them to feature properties
def extract_image_properties(img):
    # Get the datetime of the image
    datetime = img.get('system:time_start')
    # Convert the timestamp to a human-readable date string
    date = ee.Date(datetime).format(params['datetimeFormat'])
    
    # Create a dictionary with the image properties
    img_props = ee.Dictionary({
        'datetime': date,
        'timestamp': datetime
    })
    
    return img_props

def zonal_stats(ic, fc, params):
    """
    Perform zonal statistics on an image collection and return the results.
    """
    # Initialize internal parameters
    _params = {
        'reducer': ee.Reducer.mean(),
        'scale': None,
        'crs': None,
        'bands': None,
        'datetimeName': 'datetime',
        'datetimeFormat': 'YYYY-MM-dd HH:mm:ss'
    }
    
    # Update parameters with provided values
    if params:
        _params.update({k: v for k, v in params.items() if v is not None})
    
    def process_image(img):
        # Select specified bands
        img = ee.Image(img.select(_params['bands']))
        
        # Get image properties
        img_props = extract_image_properties(img)
        
        # Filter feature collection based on image geometry
        fc_sub = fc.filterBounds(img.geometry())
        
        # Reduce image by regions
        reduced = img.reduceRegions(
            collection=fc_sub,
            reducer=_params['reducer'],
            scale=_params['scale'],
            crs=_params['crs'],
            tileScale=1
        )
        
        # Add metadata to each feature
        def add_properties(feature):
            return feature.set(img_props)
        
        return reduced.map(add_properties)
    
    # Process each image in the collection
    results = ic.map(process_image).flatten()
    
    return results

def reduce_image(img, fc, _params):
    # Select bands
    img = img.select(_params['bands'])
    
    # Extract image properties
    img_props = extract_image_properties(img)
    
    # Subset points that intersect the given image
    fc_sub = fc.filterBounds(img.geometry())
    
    # Reduce the image by regions
    reduced = img.reduceRegions(**{
        'collection': fc_sub,
        'reducer': _params['reducer'],
        'scale': _params['scale'],
        'crs': _params['crs'],
        'tileScale': 1  # Change to 4 if scaling errors occur
    })
    
    # Add metadata to each feature
    return reduced.map(lambda f: f.set(img_props))

# Function to extract data from covariates image and sample locations
def extract_data_func_v3(covariates_image, sample_locations):
    extracted = covariates_image.reduceRegions(**{
        'collection': sample_locations,
        'reducer': ee.Reducer.mean(),
        'scale': params['dataExtractionScale'],
        'tileScale': 4
    })
    
    return extracted


##### Functions for Landsat data ##################################

# Function to apply scaling factors for Landsat 4,5,7 SR
def scale_L457_sr(image):
    # Apply scaling factors to the appropriate bands
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_band = image.select('ST_B6').multiply(0.00341802).add(149.0)
    
    # Replace original bands with the scaled ones
    return image.addBands(optical_bands, None, True).addBands(thermal_band, None, True).copyProperties(image, ['system:time_start'])

# Function to apply scaling factors for Landsat 8 SR
def scale_L8_sr(image):
    # Apply scaling factors to the appropriate bands
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_band = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    
    # Replace original bands with the scaled ones
    return image.addBands(optical_bands, None, True).addBands(thermal_band, None, True).copyProperties(image, ['system:time_start'])

# Cloud masking function for Landsat 4,5,7
def mask_L457_sr(image):
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)
    
    # Apply the masks
    return image.updateMask(qa_mask).updateMask(saturation_mask).copyProperties(image, ['system:time_start'])

# Cloud masking function for Landsat 8
def mask_L8_sr(image):
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)
    
    # Apply the masks
    return image.updateMask(qa_mask).updateMask(saturation_mask).copyProperties(image, ['system:time_start'])

# Function to add vegetation indices bands
def add_optic_indices(image):
    ndvi = image.normalizedDifference(['nir', 'red']).rename('ndvi')
    return image.addBands([ndvi])



In [ ]:
# Define a function for processing one site
def PROCESS_ONE_SITE(demo_site, export_mode, debug, name):

    
    # Determine based on location of demo_site
    donor_aoi = GAUL0.filterBounds(demo_site.geometry())

    # Define parameters

    params['debug'] = debug
    params['projectAoi'] = demo_site
    params['donorAoi'] = donor_aoi

    
    # Adjust start and end date if debug mode is enabled
    if params['debug']:
        params['startDate'] = '2012-01-01'
        params['endDate'] = '2014-12-31'

    # Filter for Landsat data based on spatial and temporal constraints
    filterSpace = ee.Filter.bounds(params['donorAoi'])
    filterTime = ee.Filter.date(params['startDate'], params['endDate'])
    filterOverallCloud = ee.Filter.lt('CLOUD_COVER', params['sceneOverallCloudThres'])

    # Load Landsat collections
    L5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filter(filterOverallCloud).filter(filterSpace).filter(filterTime)
    L7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").filter(filterOverallCloud).filter(filterSpace).filter(filterTime)
    L8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").filter(filterOverallCloud).filter(filterSpace).filter(filterTime)

    # Load WorldCover and Land Cover datasets
    worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select('Map')
    annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
    five_year = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/five-years-map")

    #  Other data layers for covariates (TO ADD AS NEEDED)
    modis_npp = ee.ImageCollection("MODIS/061/MOD17A3HGF").select('Npp')

    elevation = ee.Image("USGS/SRTMGL1_003")

    gdp_ppp = ee.Image("projects/sat-io/open-datasets/GRIDDED_HDI_GDP/GDP_PPP_1990_2015_5arcmin_v2")

    gdp_per_capita = ee.Image("projects/sat-io/open-datasets/GRIDDED_HDI_GDP/GDP_per_capita_PPP_1990_2015_v2")

    # Add other layers as needed...

    # Other data layers for masking
    wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

    ####################################################################################
    ##### Create control areas ##################################     
    # ####################################################################################  

    # Calculate the mode of land cover in the project area
    lcModeWC = worldcover.reduceRegion(
        reducer=ee.Reducer.mode(),
        geometry=params['projectAoi'].geometry(),
        maxPixels=1e13
    )
    selLcId = ee.Number(lcModeWC.get('Map')).int()

    # Create dictionary for land cover classes and remap persistent land cover
    lcIdMatchingDict = ee.Dictionary({
        '10': [51, 52, 61, 62, 71, 72, 81, 82, 91, 92],
        '20': [120, 121, 122],
        '30': [130, 153],
        '40': [10, 11, 12, 20],
        '50': [190],
        '60': [200, 201, 202, 150, 152],
        '70': [220],
        '80': [210],
        '90': [181, 182, 183, 184, 186],
        '95': [185],
        '100': [140]
    })
    
    selLcIdPersistent = lcIdMatchingDict.get(ee.Number(selLcId).int().format())

    # Apply the land cover mask
    lcMaskWC = worldcover.updateMask(worldcover.eq(selLcId)).clip(params['donorAoi'])

    # Apply the land cover mask based on persistent LC (if enabled)
    annual_mos = annual.mosaic().clip(params['donorAoi'])
    five_year_mos = five_year.mosaic().clip(params['donorAoi'])
    all_available_years_mos = ee.Image.cat([annual_mos, five_year_mos])
    
    is_LC = all_available_years_mos.remap(selLcIdPersistent, ee.List.repeat(1, ee.List(selLcIdPersistent).size()))
    always_LC = is_LC.reduce(ee.Reducer.min()).eq(1).selfMask()

    lcMaskPersistent = always_LC

    # Use static or persistent land cover mask
    lcMaskFinal = lcMaskPersistent if params['usePersistentLC'] else lcMaskWC

    # Exclude project area buffer from donor areas
    projectAreaBuffer = params['projectAoi'].map(lambda ft: ft.buffer(params['projectAreaBuffer']).set('dummy', 1))


    projectBufferMask = ee.Image(projectAreaBuffer.reduceToImage(properties=['dummy'], reducer=ee.Reducer.first())) \
        .mask() \
        .neq(1) \
        .selfMask() \
        .clip(params['donorAoi'])


    # Final donor areas mask
    donorAreasFinal = lcMaskFinal.updateMask(projectBufferMask)

    # Mask for project area
    projectAreaMask = lcMaskFinal 

    # Generate control points in the donor areas
    controlSamplesPoints = donorAreasFinal.stratifiedSample(
        numPoints=params['numberOfControlUnits'],
        region=params['donorAoi'],
        scale=params['dataExtractionScale'],
        seed=42,
        geometries=True
    )

    # Define a function to buffer points
    def bufferPoints(radius, bounds=False):
        def wrapper(pt):
            pt = ee.Feature(pt)
            return pt.buffer(radius).bounds() if bounds else pt.buffer(radius)
        return wrapper
    

    # Buffer control points to create control areas
    controlRadius = 3500  ## here here to automate
    controlSamplesBuffered = controlSamplesPoints.map(bufferPoints(controlRadius))

    # Add unique ID to control units
    controlSamplesBuffered = controlSamplesBuffered.map(lambda ft: ft.set('id', ft.id()))

    
    ####################################################################################
    ##### Data extraction for Landsat ##################################     
    # ####################################################################################  

    # Filter the Landsat collections based on cloud cover, space, and time filters
    L5 = L5.filter(filterOverallCloud).filter(filterSpace).filter(filterTime)
    L7 = L7.filter(filterOverallCloud).filter(filterSpace).filter(filterTime)
    L8 = L8.filter(filterOverallCloud).filter(filterSpace).filter(filterTime)

    # Band mappings for Landsat collections
    l5Bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
    l5names = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']

    l7Bands = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
    l7names = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']

    l8Bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    l8names = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']

    # Apply scaling functions (assuming `scaleL457sr` and `scaleL8sr` are predefined functions)
    L5 = L5.map(scale_L457_sr)
    L7 = L7.map(scale_L457_sr)
    L8 = L8.map(scale_L8_sr)

    # Apply cloud mask and rename bands based on filtering per pixel cloud
    if params['filterPerPixelCloud']:
        L5 = L5.map(mask_L457_sr).select(l5Bands, l5names)
        L7 = L7.map(mask_L457_sr).select(l7Bands, l7names)
        L8 = L8.map(mask_L8_sr).select(l8Bands, l8names)
    else:
        L5 = L5.select(l5Bands, l5names)
        L7 = L7.select(l7Bands, l7names)
        L8 = L8.select(l8Bands, l8names)

    # Apply vegetation indices (assuming `addOpticIndices` is predefined)
    L5 = L5.map(add_optic_indices)
    L7 = L7.map(add_optic_indices)
    L8 = L8.map(add_optic_indices)

    # Merge Landsat sensors based on the parameter
    if params['mergeSensors']:
        landsatColFinal = L5.merge(L7).merge(L8).select('ndvi')
    else:
        landsatColFinal = L7.select('ndvi')  # Use Landsat-7 data only if not merging



    # Function to create annual Landsat composites

    def make_yearly_landsat_composite(year):
        start = ee.Date.fromYMD(year, 1, 1)
        end = start.advance(1, 'year')
        filter_year = ee.Filter.date(start, end)
        
        landsat_col_final_year = landsatColFinal.filter(filter_year)
        reduced = landsat_col_final_year.reduce(params['annualReducer'])
        
        old_names = reduced.bandNames()
        new_names = old_names.map(lambda element: ee.String(element).cat('_').cat(ee.Number(year).int().format()))
        
        return reduced.select(old_names, new_names)

    # Get the list of years
    year_start = ee.Date(params['startDate']).get('year').getInfo()
    year_end = ee.Date(params['endDate']).get('year').getInfo()
    years = ee.List.sequence(year_start, year_end)

    # Map the function over the list of years
    landsat_annual_composite = years.map(make_yearly_landsat_composite)
    landsat_annual_composite = ee.ImageCollection(landsat_annual_composite)

    # Convert the image collection into a multi-band image
    landsat_annual_composite_as_bands = landsat_annual_composite.toBands()

    # Masking based on project area
    landsat_col_final_for_project_area = landsatColFinal.map(lambda image: image.updateMask(projectAreaMask))
    landsat_annual_composite_as_bands_for_project_area = landsat_annual_composite_as_bands.updateMask(projectAreaMask)

    # Masking based on donor areas
    landsat_col_final = landsatColFinal.map(lambda image: image.updateMask(donorAreasFinal))
    landsat_annual_composite_as_bands = landsat_annual_composite_as_bands.updateMask(donorAreasFinal)

    # Data extraction for Landsat
    if params['exportPerDate']:
        this_params = {
            'reducer': ee.Reducer.mean(),
            'scale': params['dataExtractionScale'],
            'bands': ['ndvi'],
            'bandsRename': ['is_NDVI'],
            'datetimeName': 'date',
            'datetimeFormat': 'YYYY-MM-dd'
        }
        
        control_units_landsat = zonal_stats(landsat_col_final, controlSamplesBuffered, this_params)
        project_area_landsat = zonal_stats(landsat_col_final_for_project_area, params['projectAoi'], this_params)
        
        if export_mode == "controls":
            export_description = f"controlUnitsLandsat_v{params['version']}_{params['scriptLink']}_{name}"
            task = ee.batch.Export.table.toDrive(
                collection=control_units_landsat,
                description=export_description,
                folder=params['gdriveFolder'],
                fileFormat='CSV'
            )
            task.start()
            
        elif export_mode == "project":
            export_description = f"projectAreaLandsat_v{params['version']}_{params['scriptLink']}_{name}"
            task = ee.batch.Export.table.toDrive(
                collection=project_area_landsat,
                description=export_description,
                folder=params['gdriveFolder'],
                fileFormat='CSV'
            )
            task.start()

    else:
        control_units_landsat = extract_data_func_v3(landsat_annual_composite_as_bands, controlSamplesBuffered)
        project_area_landsat = extract_data_func_v3(landsat_annual_composite_as_bands_for_project_area, params['projectAoi'])
        
        if export_mode == "controls":
            export_description = f"controlUnitsLandsat_v{params['version']}_{params['scriptLink']}_{name}"
            task = ee.batch.Export.table.toDrive(
                collection=control_units_landsat,
                description=export_description,
                folder=params['gdriveFolder'],
                fileFormat='CSV'
            )
            task.start()
            
        elif export_mode == "project":
            export_description = f"projectAreaLandsat_v{params['version']}_{params['scriptLink']}_{name}"
            task = ee.batch.Export.table.toDrive(
                collection=project_area_landsat,
                description=export_description,
                folder=params['gdriveFolder'],
                fileFormat='CSV'
            )
            task.start()

    ####################################################################################
    ##### Data extraction for other covariates than Landsat ##################################     
    # ####################################################################################   

    # Example of stacking covariates: MODIS NPP and elevation
    modis_npp = modis_npp.map(lambda image: image.rename(ee.String('Npp').cat(ee.Number(ee.Date(image.get('system:time_start')).get('year')).format())))

    modis_npp_as_bands = modis_npp.toBands()

    # Stack NPP and elevation
    stacked = ee.Image.cat([
        modis_npp_as_bands,
        elevation.rename('elevation')
    ])

    # Mask the stack based on project area and donor areas
    stacked_for_project_area = stacked.updateMask(projectAreaMask)
    stacked = stacked.updateMask(donorAreasFinal)

    # Extract and export data for control units
    control_units_stacked = extract_data_func_v3(stacked, controlSamplesBuffered)

    # Extract and export data for project area
    project_area_stacked = extract_data_func_v3(stacked_for_project_area, params['projectAoi'])

    if export_mode == "controls":
        export_description = f"controlUnitsStacked_v{params['version']}_{params['scriptLink']}_{name}"
        task = ee.batch.Export.table.toDrive(
            collection=control_units_stacked,
            description=export_description,
            folder=params['gdriveFolder'],
            fileFormat='CSV'
        )
        task.start()
        
    elif export_mode == "project":
        export_description = f"projectAreaStacked_v{params['version']}_{params['scriptLink']}_{name}"
        task = ee.batch.Export.table.toDrive(
            collection=project_area_stacked,
            description=export_description,
            folder=params['gdriveFolder'],
            fileFormat='CSV'
        )
        task.start()


   

# Test run

In [ ]:
# SITE_91140
# SITE_91328
# SITE_90187

In [ ]:
# Get a list of unique 'site_id' values
sites_site_id_list = ee.List(sites.distinct('site_id')
                             .reduceColumns(ee.Reducer.toList(), ['site_id'])
                             .get('list'))

# Test with the first 5 site IDs from the list
# sites_site_id_list_sel = sites_site_id_list.slice(0, 5)

# Convert the list to a Python list for looping
# sites_site_id_list_sel_py = sites_site_id_list_sel.getInfo()

# sites_site_id_list_sel_py = ['SITE_91140', 'SITE_91328', 'SITE_90187']
sites_site_id_list_sel_py = ['SITE_90187']


# Loop over each site_id and process the corresponding site
for site_id in sites_site_id_list_sel_py:

    print(f"Processing site: {site_id}")

    # Filter the sites collection for the current 'site_id'
    filtered_sites = sites.filter(ee.Filter.eq('site_id', site_id))
    
    # Call the processing function for each site
    PROCESS_ONE_SITE(filtered_sites, "project", False, site_id)

# Real run

In [ ]:
not_run = True

if not not_run:

    ## Todo: filter the sites !

    # Get a list of unique 'site_id' values
    sites_site_id_list = ee.List(sites.distinct('site_id')
                                .reduceColumns(ee.Reducer.toList(), ['site_id'])
                                .get('list'))

    # Convert the list to a Python list for looping
    sites_site_id_list = sites_site_id_list.getInfo()

    # Loop over each site_id and process the corresponding site
    for site_id in sites_site_id_list:

        print(f"Processing site: {site_id}")

        # Filter the sites collection for the current 'site_id'
        filtered_sites = sites.filter(ee.Filter.eq('site_id', site_id))
        
        # Call the processing function for each site
        PROCESS_ONE_SITE(filtered_sites, "project", False, site_id)